# Optimización de Hiperparámetros con Algoritmos Genéticos

Cuaderno listo para **Google Colab** o **JupyterLab**.

Incluye:
- codificación genética de hiperparámetros,
- decodificación a parámetros reales,
- evaluación con validación cruzada,
- selección por torneo,
- cruce y mutación,
- gráficos de convergencia y distribución de fitness.


In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris
from sklearn.model_selection import cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline


## Datos del problema


In [ ]:
X, y = load_iris(return_X_y=True)

tam_poblacion = 60
n_generaciones = 60
prob_cruce = 0.8
prob_mutacion = 0.15
tam_torneo = 3

random.seed(42)
np.random.seed(42)


## Diccionarios de decodificación


In [ ]:
weights_map = {0: 'uniform', 1: 'distance'}
metric_map = {0: 'euclidean', 1: 'manhattan'}


## Representación y fitness


In [ ]:
def crear_individuo():
    return [
        random.randint(1, 20),
        random.randint(0, 1),
        random.randint(0, 1)
    ]

def decodificar(individuo):
    return {
        'n_neighbors': int(individuo[0]),
        'weights': weights_map[int(individuo[1])],
        'metric': metric_map[int(individuo[2])]
    }

def fitness(individuo):
    params = decodificar(individuo)
    modelo = Pipeline([
        ('scaler', StandardScaler()),
        ('knn', KNeighborsClassifier(
            n_neighbors=params['n_neighbors'],
            weights=params['weights'],
            metric=params['metric']
        ))
    ])
    scores = cross_val_score(modelo, X, y, cv=5, scoring='accuracy')
    return scores.mean()


## Operadores genéticos


In [ ]:
def seleccion_torneo(poblacion):
    candidatos = random.sample(poblacion, tam_torneo)
    return max(candidatos, key=fitness).copy()

def cruce_un_punto(padre1, padre2):
    if random.random() < prob_cruce:
        punto = random.randint(1, len(padre1) - 1)
        hijo1 = padre1[:punto] + padre2[punto:]
        hijo2 = padre2[:punto] + padre1[punto:]
        return hijo1, hijo2
    return padre1.copy(), padre2.copy()

def mutar(individuo):
    mutado = individuo.copy()
    if random.random() < prob_mutacion:
        mutado[0] = random.randint(1, 20)
    if random.random() < prob_mutacion:
        mutado[1] = random.randint(0, 1)
    if random.random() < prob_mutacion:
        mutado[2] = random.randint(0, 1)
    return mutado


## Ejecución del algoritmo genético


In [ ]:
poblacion = [crear_individuo() for _ in range(tam_poblacion)]

mejores_fitness = []
promedios_fitness = []
mejor_individuo_global = None
mejor_fit_global = -float('inf')

for generacion in range(n_generaciones):
    nueva_poblacion = []
    while len(nueva_poblacion) < tam_poblacion:
        padre1 = seleccion_torneo(poblacion)
        padre2 = seleccion_torneo(poblacion)
        hijo1, hijo2 = cruce_un_punto(padre1, padre2)
        hijo1 = mutar(hijo1)
        hijo2 = mutar(hijo2)
        nueva_poblacion.append(hijo1)
        if len(nueva_poblacion) < tam_poblacion:
            nueva_poblacion.append(hijo2)
    poblacion = nueva_poblacion
    fitness_poblacion = [fitness(ind) for ind in poblacion]
    mejor_generacion = max(poblacion, key=fitness)
    mejor_fit = fitness(mejor_generacion)
    promedio_fit = np.mean(fitness_poblacion)
    mejores_fitness.append(mejor_fit)
    promedios_fitness.append(promedio_fit)
    if mejor_fit > mejor_fit_global:
        mejor_fit_global = mejor_fit
        mejor_individuo_global = mejor_generacion.copy()

mejores_params = decodificar(mejor_individuo_global)
print('Mejor cromosoma encontrado:', mejor_individuo_global)
print('Mejores hiperparámetros:', mejores_params)
print('Accuracy promedio CV:', round(mejor_fit_global, 4))


## Curva de convergencia


In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(mejores_fitness, label='Mejor fitness')
plt.plot(promedios_fitness, label='Fitness promedio')
plt.xlabel('Generación')
plt.ylabel('Accuracy')
plt.title('Evolución del algoritmo genético')
plt.legend()
plt.grid(True)
plt.show()


## Distribución de fitness final


In [ ]:
fitness_final = [fitness(ind) for ind in poblacion]
plt.figure(figsize=(8, 5))
plt.hist(fitness_final, bins=10, edgecolor='black')
plt.xlabel('Accuracy')
plt.ylabel('Frecuencia')
plt.title('Distribución de fitness en la población final')
plt.grid(True)
plt.show()


## Mejor individuo encontrado


In [ ]:
labels = ['n_neighbors', 'weights_code', 'metric_code']
values = mejor_individuo_global

plt.figure(figsize=(8, 5))
plt.bar(labels, values)
plt.xlabel('Hiperparámetros')
plt.ylabel('Valor codificado')
plt.title('Codificación del mejor individuo encontrado')
plt.grid(axis='y')
plt.show()


## Ejercicio sugerido

Modificar:
- el rango de `n_neighbors`,
- la cantidad de generaciones,
- la probabilidad de mutación,
- el modelo utilizado,

y comparar el mejor accuracy encontrado.
